Data Science Capstone - Week 4
-
## Logistic Regression and Feature Scaling ##
Keshia-Lee Martin

Predicting whether a film becomes a blockbuster, which I'm defining as earning $500 million or more worldwide. 
Using the same pre-release features as the Week 2 linear regression which are release year, genre, original language, and production countries. Logistic regression is the baseline classifier for random forest in my proposal.

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## Loading Data and Setting the Target

In [35]:
movie_box_office = pd.read_csv("movie_box_office_cv2.csv")

# dropping titles that don't have genre, country, language, votes listed
movie_box_office = movie_box_office[movie_box_office['Vote_Count'].notna()].copy()
print('Rows Remaining:', len(movie_box_office))

# target: 1 if the film made $500M or more worldwide, 0 if not
movie_box_office['Blockbuster'] = (movie_box_office['$Worldwide'] >= 500_000_000).astype(int)
print(movie_box_office['Blockbuster'].value_counts())
print('Share of blockbusters:', round(movie_box_office['Blockbuster'].mean() * 100, 1), '%')

Rows Remaining: 4830
Blockbuster
0    4598
1     232
Name: count, dtype: int64
Share of blockbusters: 4.8 %


~5% of the films in the dataset are considered blockbusters. A model that says "not a blockbuster" for every film would be ~95% accurate.

## 2. Build the pre-release features
Pre-release features, excluding Rating and Vote_Count, which is post-release.\
Rank and the domestic and foreign revenue columns, because they are pieces of the target.

In [36]:
# Feature names have no spaces, so interaction names read cleaner for later
movie_box_office = movie_box_office.rename(columns={'Science Fiction': 'Sci_Fi', 'TV Movie': 'TV_Movie'})
genre_cols = ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama',
              'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance',
              'Sci_Fi', 'TV_Movie', 'Thriller', 'War', 'Western']

In [37]:
# take all rows and only the columns whose labels fall in this range for country names
country_cols = list(movie_box_office.loc[:, 'Algeria':'Yugoslavia'].columns)

In [38]:
# Release years and also the pandemic years
movie_box_office['Years_Since_2000'] = movie_box_office['Year'] - 2000
movie_box_office['Pandemic_Release'] = movie_box_office['Year'].isin([2020, 2021]).astype(int)

In [39]:
# The 10 most common countries, plus how many countries are on the film
top_countries = {'United States of America': 'Prod_USA', 'United Kingdom': 'Prod_UK',
                 'France': 'Prod_France', 'Japan': 'Prod_Japan', 'China': 'Prod_China',
                 'Germany': 'Prod_Germany', 'Canada': 'Prod_Canada', 'South Korea': 'Prod_South_Korea',
                 'Hong Kong': 'Prod_Hong_Kong', 'India': 'Prod_India'}

# creating new columns from the top countries
production_cols = []
for country in top_countries:
    col_name = top_countries[country]
    movie_box_office[col_name] = movie_box_office[country]
    production_cols.append(col_name)
movie_box_office['Num_Countries'] = movie_box_office[country_cols].sum(axis=1)

In [40]:
# Original language: English is the baseline, so it gets no column
top_languages = {'Japanese': 'Lang_Japanese', 'Chinese (Mandarin)': 'Lang_Mandarin',
                 'French': 'Lang_French', 'Korean': 'Lang_Korean', 'Hindi': 'Lang_Hindi',
                 'German': 'Lang_German', 'Spanish': 'Lang_Spanish',
                 'Chinese (Cantonese)': 'Lang_Cantonese', 'Italian': 'Lang_Italian'}

language_cols = []
for language in top_languages:
    col_name = top_languages[language]
    movie_box_office[col_name] = (movie_box_office['Original_Language'] == language).astype(int)
    language_cols.append(col_name)
named_languages = ['English'] + list(top_languages.keys())
movie_box_office['Lang_Other'] = (~movie_box_office['Original_Language'].isin(named_languages)).astype(int)
language_cols.append('Lang_Other')

In [41]:
feature_cols = ['Years_Since_2000', 'Pandemic_Release', 'Num_Countries'] + genre_cols + language_cols + production_cols
X = movie_box_office[feature_cols]
y = movie_box_office['Blockbuster']
print('Number of pre-release features:', len(feature_cols))

Number of pre-release features: 42


## Train/Test

For the train/test split, we are going to do 70/30 and a 5-fold cross validation, like Week 2. This time both are stratified, so every split keeps the same 5% share of blockbusters. Without that, a fold could end up with very few blockbusters to learn from or test on.

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Train rows:', len(X_train), '| Blockbusters:', y_train.sum())
print('Train rows:', len(X_test), '| Test rows:', y_test.sum())

results = {}


def score_model(name, model, cv_auc):
    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = model.predict(X_test)
    caught = ((test_pred == 1) & (y_test == 1)).sum()
    results[name] = {'CV ROC AUC': round(cv_auc, 3),
                     'Test ROC AUC': round(roc_auc_score(y_test, test_prob), 3),
                     'Accuracy': round(accuracy_score(y_test, test_pred), 3),
                     'Precision': round(precision_score(y_test, test_pred, zero_division=0), 3),
                     'Recall': round(recall_score(y_test, test_pred), 3),
                     'F1': round(f1_score(y_test, test_pred, zero_division=0), 3),
                     'Films flagged': int(test_pred.sum()),
                     'Blockbusters caught': str(caught) + ' of ' + str(y_test.sum())}
    print(name, '| accuracy', results[name]['Accuracy'], '| precision', results[name]['Precision'],
          '| recall', results[name]['Recall'], '| ROC AUC', results[name]['Test ROC AUC'])
    return test_prob

Train rows: 3381 | Blockbusters: 162
Train rows: 1449 | Test rows: 70


## Baseline
The baseline predicts "not a blockbuster", for every film.

In [43]:
baseline = DummyClassifier(strategy='most_frequent')
cv = cross_val_score(baseline, X_train, y_train, cv=skfold, scoring='roc_auc').mean()
baseline.fit(X_train, y_train)
baseline_prob = score_model('Baseline (not a Blockbuster)', baseline, cv)

Baseline (not a Blockbuster) | accuracy 0.952 | precision 0.0 | recall 0.0 | ROC AUC 0.5


The baseline is 95.2% accurate but catches 0 of the 70 blockbusters in the test set. Accuracy alone is misleading here, so I focus on precision, recall, and ROC AUC.

## Feature Scaling
Most features are 0/1 flags, but Years_Since_2000 runs from 0 to 24 and Num_Countries from 0 to 10. Scaling puts every feature on a similar range.

In [44]:
show_cols = ['Years_Since_2000', 'Num_Countries', 'Prod_USA', 'Animation']

standard = StandardScaler().fit(X_train)
minmax = MinMaxScaler().fit(X_train)
X_train_standard = pd.DataFrame(standard.transform(X_train), columns=feature_cols, index=X_train.index)
X_train_minmax = pd.DataFrame(minmax.transform(X_train), columns=feature_cols, index=X_train.index)

print('Raw')
print(X_train[show_cols].describe().round(2))
print()
print('StandardScaler')
print(X_train_standard[show_cols].describe().round(2))
print()
print('MinMaxScaler')
print(X_train_minmax[show_cols].describe().round(2))

Raw
       Years_Since_2000  Num_Countries  Prod_USA  Animation
count           3381.00        3381.00   3381.00    3381.00
mean              11.88           1.49      0.66       0.10
std                7.24           0.88      0.47       0.31
min                0.00           0.00      0.00       0.00
25%                6.00           1.00      0.00       0.00
50%               12.00           1.00      1.00       0.00
75%               18.00           2.00      1.00       0.00
max               24.00          10.00      1.00       1.00

StandardScaler
       Years_Since_2000  Num_Countries  Prod_USA  Animation
count           3381.00        3381.00   3381.00    3381.00
mean               0.00           0.00      0.00       0.00
std                1.00           1.00      1.00       1.00
min               -1.64          -1.69     -1.39      -0.34
25%               -0.81          -0.56     -1.39      -0.34
50%                0.02          -0.56      0.72      -0.34
75%                0

StandardScaler centers every column at 0 with a standard deviation of 1. MinMaxScaler keeps the 0/1 flags the same and squeezes Years_Since_2000 and Num_Countries into 0 to 1.

## Logistic Regression: No Scaling vs Scaled
Same model three ways.

In [45]:
lr_raw = LogisticRegression(max_iter=1000)
lr_standard = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
lr_minmax = Pipeline([('scale', MinMaxScaler()), ('model', LogisticRegression(max_iter=1000))])

scaling_models = [('Logistic, no scaling', lr_raw),
                  ('Logistic, StandardScaler', lr_standard),
                  ('Logistic, MinMaxScaler', lr_minmax)]
for item in scaling_models:
    name = item[0]
    model = item[1]
    cv = cross_val_score(model, X_train, y_train, cv=skfold, scoring='roc_auc').mean()
    model.fit(X_train, y_train)
    score_model(name, model, cv)

print()
print('Solver steps, no scaling:', lr_raw.n_iter_[0])
print('Solver steps, StandardScaler:', lr_standard.named_steps['model'].n_iter_[0])
print('Solver steps, MinMaxScaler:', lr_minmax.named_steps['model'].n_iter_[0])

Logistic, no scaling | accuracy 0.948 | precision 0.375 | recall 0.129 | ROC AUC 0.903
Logistic, StandardScaler | accuracy 0.947 | precision 0.394 | recall 0.186 | ROC AUC 0.905
Logistic, MinMaxScaler | accuracy 0.947 | precision 0.267 | recall 0.057 | ROC AUC 0.899

Solver steps, no scaling: 98
Solver steps, StandardScaler: 26
Solver steps, MinMaxScaler: 24


Scaling did not change how well the model ranks films. The cross-validated ROC AUC was 0.886 to 0.887 for all three. It did change how hard the solver had to work. Without scaling it needed 98 steps, close to the default limit of 100. With scaling it needed 24 to 26.

## Balanced Class Weights and Tuning C
Makes each missed blockbuster count about 20 times more than a missed regular film, so the model stops ignoring the rare class. `C` is the penalty setting.

In [46]:
lr_pipe = Pipeline([('scale', StandardScaler()),
                    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
lr_grid = {'model__C': [0.001, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]}
lr_search = GridSearchCV(lr_pipe, lr_grid, cv=skfold, scoring='roc_auc')
lr_search.fit(X_train, y_train)
print('Best C:', lr_search.best_params_['model__C'])
best_model = lr_search.best_estimator_
best_prob = score_model('Logistic, StandardScaler, balanced', best_model, lr_search.best_score_)

Best C: 0.01
Logistic, StandardScaler, balanced | accuracy 0.832 | precision 0.205 | recall 0.857 | ROC AUC 0.901


## Results

In [47]:
results_table = pd.DataFrame.from_dict(results, orient='index')
results_table

,CV ROC AUC,Test ROC AUC,Accuracy,Precision,Recall,F1,Films flagged,Blockbusters caught
Baseline (not a Blockbuster),0.500,0.500,0.952,0.000,0.000,0.000,0,0 of 70
"Logistic, no scaling",0.887,0.903,0.948,0.375,0.129,0.191,24,9 of 70
"Logistic, StandardScaler",0.886,0.905,0.947,0.394,0.186,0.252,33,13 of 70
"Logistic, MinMaxScaler",0.887,0.899,0.947,0.267,0.057,0.094,15,4 of 70
"Logistic, StandardScaler, balanced",0.886,0.901,0.832,0.205,0.857,0.331,293,60 of 70


## Confusion Matrix (balanced model, test set)

In [48]:
best_pred = best_model.predict(X_test)
pd.crosstab(y_test, best_pred, rownames=['Actual'], colnames=['Predicted'])

Predicted,0,1
Actual,,
0,1146,233
1,10,60


Of the 70 blockbusters in the test set, the model missed 10. Of the 1,379 other films, it wrongly flagged 233.